# Fine-tune the MSME & NABARD QA model on a free GPU

Works on **Google Colab** (free T4) and **Kaggle** (free T4, 30 GPU hours/week).

- **Colab:** `Runtime → Change runtime type → T4 GPU`, then `Runtime → Run all`.
- **Kaggle:** `File → Import Notebook`, then in the right panel set **Accelerator: GPU T4 x2** and turn **Internet on** (phone verification needed once), then `Run All`.

The result is `adapter.zip` (~70 MB, the LoRA weights). Your PC applies it to Qwen2.5-1.5B at load time (`scripts/serve_local_model.ps1`), so no merged model or GPU is needed to use it.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
%pip install -q "transformers>=4.46" "peft>=0.13" "trl>=0.19" "datasets>=3.0" "bitsandbytes>=0.44" "accelerate>=1.0"
# Colab preinstalls torchao 0.10, which recent PEFT rejects when merging the adapter
# ("only versions above 0.16.0 are supported"). It isn't needed here, so remove it.
%pip uninstall -q -y torchao

In [ ]:
import os, shutil
WORK = '/kaggle/working/nabard' if os.path.isdir('/kaggle/working') else '/content/nabard'
os.makedirs(f'{WORK}/training', exist_ok=True)
os.makedirs(f'{WORK}/data/training', exist_ok=True)
os.chdir(WORK)

try:
    from google.colab import files  # Colab
    uploaded = files.upload()        # pick all 3 files with Ctrl+click
    names = list(uploaded)
except ImportError:                  # Kaggle: add the files as a dataset (they appear under /kaggle/input/...)
    names = []

# also accept files dragged into the Files panel, or a Kaggle input dataset
search = ['.', '/content'] + [os.path.join(r, '') for r, _, _ in os.walk('/kaggle/input')]
for name in set(names) | {'finetune_lora.py', 'sft_train.jsonl', 'sft_val.jsonl'}:
    dest = ('training/' if name.endswith('.py') else 'data/training/') + name
    for d in search:
        src = os.path.join(d, name)
        if os.path.exists(src) and not os.path.exists(dest):
            shutil.copy(src, dest)

need = ['training/finetune_lora.py', 'data/training/sft_train.jsonl', 'data/training/sft_val.jsonl']
missing = [p for p in need if not os.path.exists(p)]
if missing:
    raise SystemExit(f'Missing {missing}. Re-run this cell and select all 3 files (Ctrl+click), '
                     'or drag them into the Files panel and re-run.')
!python training/finetune_lora.py --validate-only

In [ ]:
# QLoRA fine-tune + merge the adapter into full weights (uses the GPU; ~30-45 min on a T4)
# Expect "precision: fp16" on a T4, then a loss line every 10 steps.
# Re-running after a failed merge only redoes the merge (the trained adapter is kept).
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'  # Kaggle "T4 x2": one GPU; splitting a 4-bit model across two is slower
WORK = '/kaggle/working/nabard' if os.path.isdir('/kaggle/working') else '/content/nabard'
os.chdir(WORK)
assert os.path.exists('training/finetune_lora.py'), 'Run the previous cell first - the training files are missing'
assert '--merge-only' in open('training/finetune_lora.py').read(), \
    'This is the old training script - upload the latest finetune_colab_selfcontained.ipynb'
if os.path.exists('out/nabard-qa-lora/adapter/adapter_config.json'):
    print('Trained adapter found - skipping training, merging only')
    !python training/finetune_lora.py --merge-only --out out/nabard-qa-lora
else:
    !python training/finetune_lora.py --base Qwen/Qwen2.5-1.5B-Instruct --epochs 2 --merge --out out/nabard-qa-lora
if not os.path.isdir('out/nabard-qa-lora/adapter'):
    raise SystemExit('Training FAILED - scroll up to the Traceback above and paste its last lines.')
print('\nTraining finished: out/nabard-qa-lora/adapter is ready.')

In [ ]:
# Quick check: answer one validation question with the merged model
import json, torch, os
from transformers import AutoModelForCausalLM, AutoTokenizer
WORK = '/kaggle/working/nabard' if os.path.isdir('/kaggle/working') else '/content/nabard'
os.chdir(WORK)
path = 'out/nabard-qa-lora/merged'
if not os.path.isdir(path):
    print('No merged model (merge step failed?) - skipping the check; the adapter can still be downloaded below.')
else:
    tok = AutoTokenizer.from_pretrained(path)
    model = AutoModelForCausalLM.from_pretrained(path, dtype=torch.float16, device_map='auto')
    row = json.loads(open('data/training/sft_val.jsonl', encoding='utf-8').readline())
    prompt = tok.apply_chat_template(row['messages'][:-1], tokenize=False, add_generation_prompt=True)
    ids = tok(prompt, return_tensors='pt').to(model.device)
    out = model.generate(**ids, max_new_tokens=300, do_sample=False)
    print('QUESTION:', row['messages'][1]['content'].rsplit('Question:', 1)[-1].strip())
    print('\nMODEL:   ', tok.decode(out[0][ids['input_ids'].shape[1]:], skip_special_tokens=True))
    print('\nTARGET:  ', row['messages'][-1]['content'])
    del model; torch.cuda.empty_cache()

In [ ]:
# Download the trained adapter (~70 MB)
import os, shutil
WORK = '/kaggle/working/nabard' if os.path.isdir('/kaggle/working') else '/content/nabard'
os.chdir(f'{WORK}/out/nabard-qa-lora')
assert os.path.exists('adapter/adapter_config.json'), 'No adapter - did the training cell succeed?'
zip_path = shutil.make_archive(os.path.dirname(WORK) + '/adapter', 'zip', '.', 'adapter')
print(f'{zip_path}: {os.path.getsize(zip_path) / 1e6:.0f} MB')
try:
    from google.colab import files  # Colab: browser download
    files.download(zip_path)
except ImportError:                  # Kaggle: it's in /kaggle/working
    print('Kaggle: open the Output panel on the right (/kaggle/working) and download adapter.zip')

## Use it on your PC (CPU only)

1. Unzip `adapter.zip` into `out/nabard-qa-lora/` in the project.
2. Convert it for llama.cpp:
   `python llama.cpp/convert_lora_to_gguf.py out/nabard-qa-lora/adapter --base <Qwen2.5-1.5B-Instruct config dir> --outtype f16 --outfile training/ollama/nabard-qa-lora-f16.gguf`
3. Start the model: `powershell -ExecutionPolicy Bypass -File scripts\serve_local_model.ps1` (port 8001, model `nabard-qa-lora`). The app's `local` provider uses it, e.g. `NABARD_QA_LLM_CHAIN=groq,gemini,local`.